## Ingestion Notebook

### Creating source schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze_src;

In [0]:
import requests
import json
import time
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F

### Set API key and stock symbols

In [0]:
API_KEY = dbutils.secrets.get(scope="alpha-vantage-scope", key="api-key")
symbols = ["AAPL", "MSFT", "GOOGL", "TSLA"]

### Creating API helper function

Instead of writing the API call logic again and again, I created small helper functions for each endpoint.  
This makes the notebook cleaner and easier to maintain. Each function is responsible for one type of data: daily stock history, latest quote, or company overview.

In [0]:
def get_time_series_daily(symbol: str) -> dict:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey": API_KEY,
        "outputsize": "compact"
    }
    response = requests.get(url, params=params, timeout=30)
    data = response.json()

    if "Information" in data or "Error Message" in data:
        raise ValueError(f"TIME_SERIES_DAILY API error for {symbol}: {data}")

    if "Time Series (Daily)" not in data:
        raise ValueError(f"Unexpected daily response for {symbol}: {data}")

    return data


def get_global_quote(symbol: str) -> dict:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "GLOBAL_QUOTE",
        "symbol": symbol,
        "apikey": API_KEY
    }
    response = requests.get(url, params=params, timeout=30)
    data = response.json()

    if "Information" in data or "Error Message" in data:
        raise ValueError(f"GLOBAL_QUOTE API error for {symbol}: {data}")

    return data


def get_company_overview(symbol: str) -> dict:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "OVERVIEW",
        "symbol": symbol,
        "apikey": API_KEY
    }
    response = requests.get(url, params=params, timeout=30)
    data = response.json()

    if "Information" in data or "Error Message" in data:
        raise ValueError(f"OVERVIEW API error for {symbol}: {data}")

    return data

### Reading latest processed daily date per symbol

For daily stock history, I do not want to insert the same dates again and again.  
So here I read the existing landing table and capture the latest trading date already available for each symbol. This helps me load only the new daily rows from the API.

In [0]:
existing_daily_df = spark.table("bronze_src.daily_stock_incoming")

latest_daily_by_symbol = {
    row["symbol"]: row["max_trading_date"]
    for row in (
        existing_daily_df.groupBy("symbol")
        .agg(F.max("trading_date").alias("max_trading_date"))
        .collect()
    )
}

### Create containers for the incoming data

Before writing anything into Delta tables, I am first collecting the API results in Python lists.  
I am keeping separate lists for daily stock history, quote snapshots, and company information so that each dataset can later be written into its own landing table.

In [0]:
daily_rows = []
quote_rows = []
company_rows = []

### Fetch and organize data for each stock symbol

In this block, I loop through each stock symbol and call the required API endpoints.  
For daily stock data, I only keep rows for dates that are newer than what is already stored.  
For quotes and company overview, I keep a fresh snapshot for every run. I also added a short pause between API calls to avoid hitting the API rate limit.

In [0]:
for symbol in symbols:
    api_pull_ts = datetime.utcnow()

    # -------------------------
    # Daily stock: append only NEW trading_date rows
    # -------------------------
    daily_data = get_time_series_daily(symbol)
    meta = daily_data.get("Meta Data", {})
    ts_data = daily_data.get("Time Series (Daily)", {})

    last_loaded_date = latest_daily_by_symbol.get(symbol)

    for trading_date_str, values in ts_data.items():
        trading_date = datetime.strptime(trading_date_str, "%Y-%m-%d").date()

        if last_loaded_date is None or trading_date > last_loaded_date:
            daily_rows.append(Row(
                symbol=symbol,
                trading_date=trading_date,
                open_price=float(values.get("1. open")),
                high_price=float(values.get("2. high")),
                low_price=float(values.get("3. low")),
                close_price=float(values.get("4. close")),
                volume=int(values.get("5. volume")),
                source_last_refreshed=meta.get("3. Last Refreshed"),
                source_timezone=meta.get("5. Time Zone"),
                api_pull_ts=api_pull_ts,
                raw_json=json.dumps(values)
            ))

    time.sleep(12)

    # -------------------------
    # Quote snapshot: append latest snapshot each run
    # -------------------------
    quote_data = get_global_quote(symbol)
    q = quote_data.get("Global Quote", {})

    quote_rows.append(Row(
        symbol=symbol,
        price=float(q.get("05. price")) if q.get("05. price") else None,
        volume=int(q.get("06. volume")) if q.get("06. volume") else None,
        latest_trading_day=datetime.strptime(q.get("07. latest trading day"), "%Y-%m-%d").date() if q.get("07. latest trading day") else None,
        previous_close=float(q.get("08. previous close")) if q.get("08. previous close") else None,
        change=float(q.get("09. change")) if q.get("09. change") else None,
        change_percent=q.get("10. change percent"),
        api_pull_ts=api_pull_ts,
        raw_json=json.dumps(quote_data)
    ))

    time.sleep(12)

    # -------------------------
    # Company overview snapshot: append snapshot each run
    # -------------------------
    company_data = get_company_overview(symbol)

    company_rows.append(Row(
        symbol=symbol,
        name=company_data.get("Name"),
        exchange=company_data.get("Exchange"),
        sector=company_data.get("Sector"),
        industry=company_data.get("Industry"),
        market_capitalization=company_data.get("MarketCapitalization"),
        country=company_data.get("Country"),
        currency=company_data.get("Currency"),
        api_pull_ts=api_pull_ts,
        raw_json=json.dumps(company_data)
    ))

    time.sleep(12)

/home/spark-9f18b058-6b99-4055-ba49-bc/.ipykernel/1994/command-5679888009784515-1203577680:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  api_pull_ts = datetime.utcnow()


### Append only when new rows exist

In [0]:
if daily_rows:
    daily_df = spark.createDataFrame(daily_rows)
    daily_df.write.format("delta").mode("append").saveAsTable("bronze_src.daily_stock_incoming")
    print(f"Appended {len(daily_rows)} new daily stock rows")
else:
    print("No new daily stock rows to append")

if quote_rows:
    quote_df = spark.createDataFrame(quote_rows)
    quote_df.write.format("delta").mode("append").saveAsTable("bronze_src.quotes_incoming")
    print(f"Appended {len(quote_rows)} quote snapshot rows")
else:
    print("No quote snapshot rows to append")

if company_rows:
    company_df = spark.createDataFrame(company_rows)
    company_df.write.format("delta").mode("append").saveAsTable("bronze_src.company_info_incoming")
    print(f"Appended {len(company_rows)} company snapshot rows")
else:
    print("No company snapshot rows to append")

Appended 4 new daily stock rows
Appended 4 quote snapshot rows
Appended 4 company snapshot rows
